In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import time
from dotenv import load_dotenv, find_dotenv
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

print("find_dotenv() ->", find_dotenv())  
load_dotenv(find_dotenv(), override=True)

print("Loaded EMAIL:", os.getenv("EMAIL"))
print("Loaded PASSWORD:", os.getenv("PASSWORD"))

RAW_COUNT = os.getenv("LINKEDIN_LIMIT")
if RAW_COUNT is None:
    print("No LINKEDIN_LIMIT detected - will run UNLIMITED mode")
    unlimited_mode = True
else:
    RAW_COUNT = RAW_COUNT.strip()
    if RAW_COUNT.isdigit():
        LIMIT = int(RAW_COUNT)
        unlimited_mode = False
        print(f"LINKEDIN_LIMIT detected: {LIMIT} - will run LIMITED mode")
    else:
        unlimited_mode = True
        print("Invalid LINKEDIN_LIMIT - will run UNLIMITED mode")

MODE_INFO = {
    'unlimited_mode': unlimited_mode,
    'limit': LIMIT if not unlimited_mode else None
}

# Initialize driver
if 'driver' not in globals():
    print("🚀 Initializing new WebDriver instance...")
    driver = webdriver.Chrome()
    driver_instance_created = True
else:
    try:
        driver.current_url
        print("🔁 Reusing existing WebDriver session")
        driver_instance_created = False
    except:
        driver = webdriver.Chrome()
        driver_instance_created = True


# ---------- FIXED LOGIN LOGIC ----------
print("🔐 Attempting LinkedIn login...")

try:
    driver.get('https://www.linkedin.com/login')

    wait = WebDriverWait(driver, 20)

    # If already logged in
    if 'feed' in driver.current_url:
        print("✅ Already logged in")
    else:
        email = wait.until(EC.element_to_be_clickable((By.ID, 'username')))
        email.clear()
        email.send_keys(os.getenv("EMAIL"))

        password = wait.until(EC.element_to_be_clickable((By.ID, 'password')))
        password.clear()
        password.send_keys(os.getenv("PASSWORD"))

        password.submit()
        time.sleep(5)

        if 'feed' in driver.current_url or 'search' in driver.current_url:
            print("✅ Login successful")
        else:
            print("⚠️ Login may have failed - check credentials or captcha")

except Exception as e:
    print(f"❌ Login error: {e}")


print("\n" + "="*60)
print("✅ DRIVER SESSION READY")
print(f"📊 Mode: {'UNLIMITED' if unlimited_mode else 'LIMITED'}")
print(f"🌐 Current URL: {driver.current_url}")
print("💡 This WebDriver will stay open until you manually close it")
print("💡 To close the browser, run: driver.quit()")
print("="*60)


find_dotenv() -> C:\Users\USER\linkedin\test_code\.env
Loaded EMAIL: nanditharajesh2004@gmail.com
Loaded PASSWORD: Nanda@123
LINKEDIN_LIMIT detected: 10 - will run LIMITED mode
🚀 Initializing new WebDriver instance...


🔐 Attempting LinkedIn login...


✅ Login successful

✅ DRIVER SESSION READY
📊 Mode: LIMITED
🌐 Current URL: https://www.linkedin.com/feed/
💡 This WebDriver will stay open until you manually close it
💡 To close the browser, run: driver.quit()


perfect only unlimited goes on till 10 only try 3

In [2]:
import sys
import time
import os
import re
import subprocess
from datetime import datetime
from bs4 import BeautifulSoup
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# Reuse the same driver instance from previous cell
print("🔄 Attempting to reuse existing WebDriver session...")

try:
    # Check if driver exists and is valid
    current_url = driver.current_url
    print(f"✅ Successfully reused WebDriver instance")
    print(f"📄 Current URL: {current_url}")
    print(f"🏠 Driver process ID: {driver.service.process.pid if driver.service else 'N/A'}")
except NameError:
    print("❌ No existing WebDriver found. Please run Cell 1 first.")
    exit()
except Exception as e:
    print(f"❌ Existing WebDriver session is invalid: {e}")
    print("🔄 Please run Cell 1 again to initialize a new session.")
    exit()

# Get mode information from previous cell
try:
    unlimited_mode = MODE_INFO['unlimited_mode']
    LIMIT = MODE_INFO['limit']
except NameError:
    # Fallback if running this cell independently
    print("📝 MODE_INFO not found - checking environment directly...")
    RAW_COUNT = os.getenv("LINKEDIN_LIMIT")

    if RAW_COUNT is not None:
        RAW_COUNT = RAW_COUNT.strip()
        if RAW_COUNT.isdigit():
            unlimited_mode = False
            LIMIT = int(RAW_COUNT)
        else:
            unlimited_mode = True
            LIMIT = None
    else:
        unlimited_mode = True
        LIMIT = None

print(f"📊 Operating in {'UNLIMITED' if unlimited_mode else 'LIMITED'} mode")
if not unlimited_mode:
    print(f"🎯 Target limit: {LIMIT} profiles")
else:
    MIN_UNLIMITED_TARGET = 10000
    print(f"🎯 Unlimited target: {MIN_UNLIMITED_TARGET} profiles")

###########################################################################
# GLOBAL STATE
###########################################################################
if 'opened_links' not in globals():
    opened_links = set()   # Ensures no link opens more than once EVER
if 'TOTAL_SAVED' not in globals():
    TOTAL_SAVED = 0

###########################################################################
# FUNCTIONS
###########################################################################
def scroll_to_find_next_button(driver):
    """Scroll through the page to help locate the Next button"""
    print("📜 Scrolling to find Next button...")
    
    # Simple scroll to bottom
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
    time.sleep(2)
    
    # Look for next button
    selectors = [
        "//button[@aria-label='Next']",
        "//button[contains(@class, 'artdeco-pagination__button--next')]",
        "//button[.//span[contains(text(), 'Next')]]"
    ]
    
    for selector in selectors:
        buttons = driver.find_elements(By.XPATH, selector)
        for btn in buttons:
            if btn.is_displayed() and btn.is_enabled():
                print(f"✅ Next button found!")
                return btn
    
    print("❌ No Next button found")
    return None

def click_next_until_end(driver):
    try:
        print("🔍 Looking for Next button...")
        time.sleep(3)
        button = scroll_to_find_next_button(driver)
        if not button:
            return False
        if not button.is_enabled():
            return False
        
        button.click()
        time.sleep(5)
        return True
    except Exception as e:
        print(f"❌ Next button click failed: {e}")
        return False

def save_profile_page(driver, url):
    """Save a single profile page"""
    global TOTAL_SAVED
    try:
        saved_profiles_dir = "saved_profiles"
        os.makedirs(saved_profiles_dir, exist_ok=True)
        
        page_source = driver.page_source
        title = driver.title
        safe_title = re.sub(r'[^a-zA-Z0-9_-]', '_', title.split('|')[0]).strip('_')
        if not safe_title:
            safe_title = "unknown_profile"
        
        filename = f"{safe_title}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.html"
        filepath = os.path.join(saved_profiles_dir, filename)
        
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(page_source)
        
        TOTAL_SAVED += 1
        print(f"💾 Saved [{TOTAL_SAVED}] {filename}")
        return True
        
    except Exception as e:
        print(f"❌ Error saving profile: {e}")
        return False

def run_parser():
    print("🔄 Running parser on saved profiles...")
    try:
        subprocess.run(["python", "parse_linkedin_02.py", "saved_profiles"], check=True)
        print("✅ Parser completed successfully")
    except subprocess.CalledProcessError as e:
        print(f"❌ Parser failed: {e}")
    except FileNotFoundError:
        print("❌ Parser script not found: parse_linkedin_02.py")

def check_driver_health():
    """Check if the WebDriver session is still valid"""
    try:
        driver.current_url
        return True
    except:
        return False

###########################################################################
# MAIN LOOP - SIMPLE AND RELIABLE
###########################################################################

# Store the original search URL to return to
SEARCH_URL = "https://www.linkedin.com/search/results/people/?heroEntityKey=urn%3Ali%3Aorganization%3A77697892&industry=%5B%224%22%5D&keywords=Canara%20Engineering%20College&origin=FACETED_SEARCH&schoolFilter=%5B%2277697892%22%5D&sid=X76"

# Navigate to search page if not already there
if SEARCH_URL not in driver.current_url:
    print(f"🌐 Navigating to search page...")
    driver.get(SEARCH_URL)
    time.sleep(5)

page_count = 1
has_more_pages = True

print(f"\n🎯 STARTING SCRAPING SESSION")
print(f"📈 Previous total: {TOTAL_SAVED} profiles")
print(f"🔗 Unique links tracked: {len(opened_links)}")

while has_more_pages:
    # Check driver health before each iteration
    if not check_driver_health():
        print("❌ WebDriver session lost! Please run Cell 1 again.")
        break
        
    print(f"\n=== Processing Page {page_count} ===")
    print(f"📊 Progress: {TOTAL_SAVED} profiles saved")

    # Collect all profile links on current page
    profile_elements = driver.find_elements(By.CSS_SELECTOR, "a[href*='/in/'].sgFaNWsBvYRpNPPEXFFIOIIaPaSfHHpphGSmo")
    profile_urls = []
    
    # If no specific links found, fall back to general links
    if not profile_elements:
        profile_elements = driver.find_elements(By.CSS_SELECTOR, "a[href*='/in/']")
        print(f"🔄 Using fallback selector, found {len(profile_elements)} links")
    
    for elem in profile_elements:
        try:
            href = elem.get_attribute("href")
            if href and '/in/' in href:
                clean_url = href.split('?')[0].split('#')[0]
                if clean_url not in opened_links:
                    profile_urls.append(clean_url)
                    opened_links.add(clean_url)
        except:
            continue
    
    print(f"🔍 Found {len(profile_urls)} new profile links on this page")
    
    # Process each profile
    for i, profile_url in enumerate(profile_urls):
        # Check stop conditions
        if not unlimited_mode and LIMIT is not None and TOTAL_SAVED >= LIMIT:
            print(f"✅ Reached limit of {LIMIT} profiles. Stopping.")
            has_more_pages = False
            break
        elif unlimited_mode and TOTAL_SAVED >= MIN_UNLIMITED_TARGET:
            print(f"✅ Reached minimum unlimited target of {MIN_UNLIMITED_TARGET}. Stopping.")
            has_more_pages = False
            break
            
        print(f"👤 Processing profile {i+1}/{len(profile_urls)}: {profile_url}")
        
        try:
            # Save current page URL to return to
            current_page = driver.current_url
            
            # Navigate to profile
            print("  🚀 Navigating to profile...")
            driver.get(profile_url)
            time.sleep(4)  # Increased wait for profile to load
            
            # Save the profile
            if save_profile_page(driver, profile_url):
                print(f"  ✅ Successfully saved profile")
            else:
                print(f"  ❌ Failed to save profile")
            
            # Return to search results
            print("  🔄 Returning to search results...")
            driver.get(current_page)
            time.sleep(3)
            
        except Exception as e:
            print(f"  ❌ Error processing profile: {e}")
            # Try to return to search page
            try:
                print("  🚑 Attempting to recover session...")
                driver.get(SEARCH_URL)
                # Try to navigate back to the specific page
                for _ in range(page_count - 1):
                    if click_next_until_end(driver):
                        time.sleep(2)
                    else:
                        break
                time.sleep(3)
            except:
                print("  🔄 Could not recover session, starting from page 1")
                driver.get(SEARCH_URL)
                page_count = 1
                time.sleep(5)
            continue

    # Run parser after processing all profiles on this page
    print("🔄 Running parser on collected profiles...")
    run_parser()
    
    # Check stop conditions again (in case we broke from inner loop)
    if not unlimited_mode and LIMIT is not None and TOTAL_SAVED >= LIMIT:
        print(f"✅ Reached limit of {LIMIT} profiles. Stopping.")
        break
    elif unlimited_mode and TOTAL_SAVED >= MIN_UNLIMITED_TARGET:
        print(f"✅ Reached minimum unlimited target of {MIN_UNLIMITED_TARGET}. Stopping.")
        break

    # Go to next page
    print("➡️  Moving to next page...")
    if click_next_until_end(driver):
        page_count += 1
        print(f"📄 Advanced to page {page_count}")
    else:
        has_more_pages = False
        print("⏹️  No more pages available")

print("\n" + "="*60)
print("🎉 SCRAPING SESSION COMPLETED")
print(f"💾 Total profiles saved: {TOTAL_SAVED}")
print(f"🔗 Unique links processed: {len(opened_links)}")
print(f"📄 Pages processed: {page_count}")
print("="*60)
print("💡 WebDriver is STILL ACTIVE and can be reused")
print("💡 Run this cell again to continue scraping")
print("💡 Run 'driver.quit()' to manually close the browser when done")
print("="*60)

🔄 Attempting to reuse existing WebDriver session...
✅ Successfully reused WebDriver instance
📄 Current URL: https://www.linkedin.com/feed/
🏠 Driver process ID: 14252
📊 Operating in LIMITED mode
🎯 Target limit: 10 profiles
🌐 Navigating to search page...



🎯 STARTING SCRAPING SESSION
📈 Previous total: 0 profiles
🔗 Unique links tracked: 0

=== Processing Page 1 ===
📊 Progress: 0 profiles saved
🔄 Using fallback selector, found 40 links


🔍 Found 17 new profile links on this page
👤 Processing profile 1/17: https://www.linkedin.com/in/manish-a48a992a4
  🚀 Navigating to profile...


💾 Saved [1] 19__Manish_20251211_232842_574536.html
  ✅ Successfully saved profile
  🔄 Returning to search results...


👤 Processing profile 2/17: https://www.linkedin.com/in/ACoAABaqOcgBHjdTwAjPG2XQas5_mkjBnMaMaXA
  🚀 Navigating to profile...


...

👤 Processing profile 10/17: https://www.linkedin.com/in/ACoAADgDw2sB6UhIn0_IS9o01JP_Lp8cqluYHgg


  🚀 Navigating to profile...


💾 Saved [10] 19__Bhagyashree_M_20251211_233032_484193.html
  ✅ Successfully saved profile
  🔄 Returning to search results...


✅ Reached limit of 10 profiles. Stopping.
🔄 Running parser on collected profiles...
🔄 Running parser on saved profiles...


✅ Parser completed successfully
✅ Reached limit of 10 profiles. Stopping.

🎉 SCRAPING SESSION COMPLETED
💾 Total profiles saved: 10
🔗 Unique links processed: 17
📄 Pages processed: 1
💡 WebDriver is STILL ACTIVE and can be reused
💡 Run this cell again to continue scraping
💡 Run 'driver.quit()' to manually close the browser when done


In [3]:
# Close browser session
#print("Closing browser session...")
#try:
  #  driver.quit()
 #   print("Browser closed successfully.")
#except Exception as e:
 #   print(f"Error closing browser: {e}")

#print("\n=== SCRAPING SESSION COMPLETE ===")